# Lab 1.4 — Data Validation, Schema Contracts & Data Versioning

**Module 1, Lab 4 | AI/ML Intermediate Workshop | Nutanix Engineering**

---

## Why This Lab Matters

You've spent Labs 1.1–1.3 loading, cleaning, and feature-engineering data. That work is valuable — but in a production ML system, data pipelines are **brittle by default**. Schema violations, silent type changes, and missing values creep in from upstream systems constantly.

**Real scenario:** Your ETL pipeline has been running fine for weeks. One day, a Nutanix AOS upgrade changes a metric collection agent's output — `cpu_percent` is now reported as a string like `"45.2%"` instead of the float `45.2`. Your preprocessing code casts it silently or fails quietly. The model is trained on garbage. Inference crashes in production, or worse — makes bad scaling decisions for 6 hours before anyone notices.

**This is not hypothetical.** Schema drift is one of the top causes of ML system failures in production.

---

## Learning Objectives

By the end of this lab you will be able to:

1. Write manual data quality checks and understand their limitations
2. Define declarative schema validation using **Pandera**
3. Author a **data contract** that describes the expected shape, types, and business rules of a dataset
4. Understand **DVC (Data Version Control)** and its role in ML pipelines
5. Implement a lightweight **data versioning system** in pure Python
6. Track **data lineage** across a multi-step transformation pipeline
7. Generate a structured **data quality report**

---

## Connection to Software Engineering Concepts You Already Know

| ML Concept | Software Engineering Analogy |
|---|---|
| Schema Validation | Input validation / type checking |
| Data Contract | API contract / interface definition |
| Data Versioning (DVC) | Git for data artifacts |
| Data Lineage | Audit log / change tracking |
| Quality Report | CI health dashboard |

> **Instructor Note:** Take 2–3 minutes here. Ask the class: "Has anyone been burned by unexpected data changes breaking a downstream system?" Engineers from infrastructure teams almost always have a story. This framing makes the lab feel immediately relevant.

## 📦 Requirements & Troubleshooting

### Required Packages

| Package | Install Name |
|---------|-------------|
| pandera | `pandera` |
| pandas | `pandas` |
| numpy | `numpy` |

**Install all at once:**
```bash
pip install pandera pandas numpy
```

---

### ⚠️ Common Errors & Fixes

**`ModuleNotFoundError: No module named '...'`**
> Package is missing from the active Python environment.
> Fix: Run the pip install command above in a terminal, then **restart the kernel**.

**`CalledProcessError` — `--break-system-packages` / exit status 2**
> You are using a virtual environment (e.g. `myenv`) where that flag is not supported, or your pip version is old.
> Fix: Open a terminal, activate your venv (`source myenv/bin/activate`), then run `pip install <package>` without that flag.

**`Failed building wheel for <package>` / C extension errors**
> The package does not support your Python version (most common on Python 3.14).
> Fix: Switch the kernel to **Python 3.13**. Click the kernel name in the VS Code top-right corner → *Select Another Kernel* → *Python 3.13*. Then re-run.

**Packages install with no error but `ModuleNotFoundError` still appears**
> You installed into a different Python than the one the notebook is using.
> Fix: Check the kernel shown in the top-right of VS Code. Open a terminal, activate that environment, and install packages there.

**`PermissionError` or `[Errno 13]` when installing**
> Trying to install into a read-only system Python.
> Fix: Use a virtual environment — `python -m venv myenv && source myenv/bin/activate` — then install.


## Environment Setup

Install `pandera` if not already available. All other libraries are standard in the workshop environment.

In [ ]:
# Install pandera if not already available
try:
    import pandera
    print(f"pandera {pandera.__version__} already installed")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandera", "--quiet", "--break-system-packages"])
    print("pandera installed successfully")


### Import Libraries

We're importing everything we'll need for the full lab upfront.

In [ ]:
import pandas as pd
import numpy as np
import pandera as pa
from pandera import Column, DataFrameSchema, Check
import json
import os
import hashlib
import warnings
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

# Set display options for cleaner output
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.width', 120)

# Base directory for saving artifacts produced in this lab
LAB_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Lab artifacts will be saved to: {LAB_DIR}")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__} | Pandera: {pa.__version__}")

---

## Step 1: Load or Generate Dataset

We'll load the `cleaned_server_logs.csv` produced in Lab 1.2 if it exists. If you're running this lab standalone, we generate a realistic inline dataset representing Nutanix cluster node telemetry.

The dataset captures per-node metrics: host identity, log level, software component, CPU %, memory usage, disk I/O, response times, and error codes — exactly the kind of data a Nutanix infrastructure monitoring pipeline would ingest.

In [ ]:
np.random.seed(42)

def generate_server_logs(n=500):
    """Generate synthetic Nutanix cluster node telemetry logs."""
    hosts = [
        "ntnx-cvm-node-01", "ntnx-cvm-node-02", "ntnx-cvm-node-03",
        "ntnx-ahv-node-01", "ntnx-ahv-node-02", "ntnx-prism-central"
    ]
    log_levels  = ["INFO", "INFO", "INFO", "WARNING", "ERROR", "DEBUG"]
    components  = ["AOS", "AHV", "Prism", "Stargate", "Cassandra", "Zookeeper", "Curator"]
    error_codes = [None, None, None, None, "E001", "E002", "E003", "W101"]

    base_time = datetime(2024, 1, 15, 8, 0, 0, tzinfo=timezone.utc)
    timestamps = [
        (base_time.replace(tzinfo=None) + pd.Timedelta(seconds=i * 30)).strftime("%Y-%m-%d %H:%M:%S")
        for i in range(n)
    ]

    df = pd.DataFrame({
        "timestamp":       timestamps,
        "host":            np.random.choice(hosts, n),
        "log_level":       np.random.choice(log_levels, n),
        "component":       np.random.choice(components, n),
        "cpu_percent":     np.clip(np.random.normal(45, 20, n), 1, 99).round(2),
        "memory_mb":       np.clip(np.random.normal(8192, 2048, n), 512, 16384).round(1),
        "disk_io_mbps":    np.clip(np.random.exponential(50, n), 0, 500).round(2),
        "response_time_ms":np.clip(np.random.lognormal(4, 1, n), 1, 5000).round(1),
        "error_code":      np.random.choice(error_codes, n),
    })
    return df


# Try loading previously cleaned data; fall back to generation
cleaned_path = os.path.join(LAB_DIR, "cleaned_server_logs.csv")

if os.path.exists(cleaned_path):
    df = pd.read_csv(cleaned_path)
    print(f"Loaded existing dataset from: {cleaned_path}")
else:
    df = generate_server_logs(n=500)
    df.to_csv(cleaned_path, index=False)
    print(f"Generated synthetic dataset and saved to: {cleaned_path}")

print(f"\nDataset shape: {df.shape}")
print(f"\nColumn dtypes:")
print(df.dtypes.to_string())

In [ ]:
# Quick structural overview
print("=== DataFrame Info ===")
df.info()
print("\n=== First 5 rows ===")
df.head()

In [ ]:
# Basic descriptive statistics for numeric columns
df.describe()

---

## Step 2: Manual Data Quality Checks (The Way Most Teams Start)

Before we introduce purpose-built validation tooling, let's see how most engineering teams approach data quality: ad-hoc assertions and if-statements scattered across notebooks or ETL scripts.

We'll write these as a structured function so you can see what "hand-rolled" validation looks like — and understand why it doesn't scale.

> **Instructor Note:** Walk through the checks slowly. Ask: "If your pipeline runs daily, how do you know yesterday's run passed all these checks?" The answer — you don't, unless you log it. That's the core problem this lab solves.

In [ ]:
def run_quality_checks(df: pd.DataFrame) -> dict:
    """
    Manual data quality checks for server log telemetry data.
    
    Returns a dict mapping check_name -> {passed, message, details}.
    This mirrors the kind of hand-rolled validation you'd find in early-stage
    data engineering teams before adopting formal validation frameworks.
    """
    results = {}

    # -----------------------------------------------------------------------
    # CHECK 1: No nulls in required columns
    # -----------------------------------------------------------------------
    required_cols = ["host", "log_level", "component", "cpu_percent", "memory_mb"]
    missing_counts = df[required_cols].isnull().sum()
    check1_pass = missing_counts.sum() == 0
    results["no_nulls_in_required_cols"] = {
        "passed": check1_pass,
        "message": "All required columns have zero nulls" if check1_pass
                   else f"Nulls detected in: {missing_counts[missing_counts > 0].to_dict()}",
        "details": missing_counts.to_dict()
    }

    # -----------------------------------------------------------------------
    # CHECK 2: cpu_percent must be between 0 and 100 (inclusive)
    # -----------------------------------------------------------------------
    if "cpu_percent" in df.columns:
        invalid_cpu = df[(df["cpu_percent"] < 0) | (df["cpu_percent"] > 100)]
        check2_pass = len(invalid_cpu) == 0
        results["cpu_percent_range"] = {
            "passed": check2_pass,
            "message": "All cpu_percent values in [0, 100]" if check2_pass
                       else f"{len(invalid_cpu)} rows have cpu_percent out of [0, 100]",
            "details": {
                "min": float(df["cpu_percent"].min()),
                "max": float(df["cpu_percent"].max()),
                "invalid_count": len(invalid_cpu)
            }
        }
    else:
        results["cpu_percent_range"] = {"passed": False, "message": "Column 'cpu_percent' missing", "details": {}}

    # -----------------------------------------------------------------------
    # CHECK 3: log_level must be in the valid set
    # -----------------------------------------------------------------------
    valid_levels = {"ERROR", "WARNING", "INFO", "DEBUG"}
    if "log_level" in df.columns:
        actual_levels = set(df["log_level"].dropna().unique())
        unexpected = actual_levels - valid_levels
        check3_pass = len(unexpected) == 0
        results["log_level_valid_values"] = {
            "passed": check3_pass,
            "message": "All log_level values are valid" if check3_pass
                       else f"Unexpected log_level values: {unexpected}",
            "details": {
                "valid_set": sorted(valid_levels),
                "found": sorted(actual_levels),
                "unexpected": sorted(unexpected)
            }
        }
    else:
        results["log_level_valid_values"] = {"passed": False, "message": "Column 'log_level' missing", "details": {}}

    # -----------------------------------------------------------------------
    # CHECK 4: No exact duplicate rows
    # -----------------------------------------------------------------------
    n_dupes = df.duplicated().sum()
    check4_pass = n_dupes == 0
    results["no_duplicate_rows"] = {
        "passed": check4_pass,
        "message": "No duplicate rows found" if check4_pass
                   else f"{n_dupes} duplicate rows detected",
        "details": {"duplicate_count": int(n_dupes)}
    }

    # -----------------------------------------------------------------------
    # CHECK 5: memory_mb must be positive
    # -----------------------------------------------------------------------
    if "memory_mb" in df.columns:
        invalid_mem = df[df["memory_mb"] <= 0]
        check5_pass = len(invalid_mem) == 0
        results["memory_mb_positive"] = {
            "passed": check5_pass,
            "message": "All memory_mb values are positive" if check5_pass
                       else f"{len(invalid_mem)} rows have non-positive memory_mb",
            "details": {
                "min": float(df["memory_mb"].min()),
                "invalid_count": len(invalid_mem)
            }
        }

    # -----------------------------------------------------------------------
    # CHECK 6: Expected columns are all present
    # -----------------------------------------------------------------------
    expected_cols = [
        "timestamp", "host", "log_level", "component",
        "cpu_percent", "memory_mb", "disk_io_mbps", "response_time_ms", "error_code"
    ]
    missing_cols = [c for c in expected_cols if c not in df.columns]
    check6_pass = len(missing_cols) == 0
    results["expected_columns_present"] = {
        "passed": check6_pass,
        "message": "All expected columns present" if check6_pass
                   else f"Missing columns: {missing_cols}",
        "details": {"missing": missing_cols, "expected": expected_cols}
    }

    return results


# Run the checks
check_results = run_quality_checks(df)

# Pretty-print results
print("=" * 65)
print(" DATA QUALITY CHECK REPORT")
print("=" * 65)
total = len(check_results)
passed = sum(1 for r in check_results.values() if r["passed"])

for check_name, result in check_results.items():
    status = "PASS" if result["passed"] else "FAIL"
    icon   = "✓" if result["passed"] else "✗"
    print(f"  [{icon}] {status}  {check_name}")
    print(f"        → {result['message']}")

print("-" * 65)
print(f"  Result: {passed}/{total} checks passed")
print("=" * 65)

### Limitations of Manual Checks

The function above works, but notice what it *doesn't* give you:

- **No type enforcement** — it won't catch `cpu_percent` silently becoming a string `"45.2%"`
- **Schema is buried in code** — a new engineer has to read the whole function to understand the expected data shape
- **Error messages are vague** — you know *something* failed, but not which rows or why
- **No composability** — adding a new check means modifying a growing function
- **Not reusable** — written for this dataset only; you'd copy-paste for every new dataset

This is where **Pandera** comes in.

---

## Step 3: Introduction to Pandera — Declarative Schema Validation

**Pandera** lets you define *what your data should look like* as a declarative schema — separate from your transformation logic. Think of it like a typed interface definition for your DataFrame.

Benefits over manual checks:
- Type checking is first-class
- Constraints are readable and co-located in one schema object
- Error messages tell you the exact row, column, and violated constraint
- Schemas can be serialized to YAML/JSON for documentation
- Works as a decorator on functions (`@pa.check_input`, `@pa.check_output`)

> **Instructor Note:** Show the Pandera docs page briefly (https://pandera.readthedocs.io). Emphasize that this is production-grade tooling used by companies like Spotify, Lyft, and GitHub in their data pipelines. It integrates with dbt, Great Expectations, and Pydantic.

In [ ]:
# Define the canonical schema for Nutanix server log telemetry
# Each Column() call specifies: dtype, constraints (Checks), and nullability

ServerLogSchema = DataFrameSchema(
    columns={
        "timestamp": Column(
            pa.String,
            nullable=False,
            description="ISO-8601 UTC timestamp of the log event"
        ),
        "host": Column(
            pa.String,
            nullable=False,
            description="Nutanix node hostname (e.g., ntnx-cvm-node-01)"
        ),
        "log_level": Column(
            pa.String,
            checks=Check.isin(["ERROR", "WARNING", "INFO", "DEBUG"]),
            nullable=False,
            description="Severity level of the log event"
        ),
        "component": Column(
            pa.String,
            nullable=False,
            description="Nutanix software component (AOS, AHV, Prism, etc.)"
        ),
        "cpu_percent": Column(
            pa.Float,
            checks=[
                Check.greater_than_or_equal_to(0.0),
                Check.less_than_or_equal_to(100.0)
            ],
            nullable=False,
            description="CPU utilization percentage at time of log"
        ),
        "memory_mb": Column(
            pa.Float,
            checks=Check.greater_than(0.0),
            nullable=False,
            description="Memory usage in megabytes"
        ),
        "disk_io_mbps": Column(
            pa.Float,
            checks=Check.greater_than_or_equal_to(0.0),
            nullable=True,  # May be missing for some node types
            description="Disk I/O throughput in MB/s (nullable for diskless nodes)"
        ),
        "response_time_ms": Column(
            pa.Float,
            checks=Check.greater_than_or_equal_to(0.0),
            nullable=True,  # Not all events have a response time
            description="API/service response time in milliseconds"
        ),
    },
    coerce=False,     # Don't silently coerce types — fail explicitly
    strict=False,     # Allow extra columns (e.g., error_code) not in the schema
    name="ServerLogTelemetrySchema",
)

print("Schema defined: ServerLogSchema")
print(f"Columns validated: {list(ServerLogSchema.columns.keys())}")
print(f"Coerce types: {ServerLogSchema.coerce} | Strict mode: {ServerLogSchema.strict}")

In [ ]:
# Validate the clean dataset against the schema
# If validation passes, schema.validate() returns the original DataFrame unchanged

try:
    validated_df = ServerLogSchema.validate(df)
    print("Schema validation PASSED")
    print(f"Validated {len(validated_df)} rows across {len(validated_df.columns)} columns")
    print(f"Returned DataFrame shape: {validated_df.shape}")
except pa.errors.SchemaError as e:
    print(f"Schema validation FAILED:\n{e}")

---

## Step 4: Schema Validation Catching Bad Data

A schema that only validates clean data isn't very useful. Let's see Pandera's real value — catching violations with actionable error messages.

We'll introduce three common real-world violations:
1. A metric value outside its physical range (`cpu_percent = 150`)
2. An unexpected categorical value (`log_level = "CRITICAL"` — not in the allowed set)
3. A null in a required column (`host = None`)

> **Instructor Note:** Point out that these violations happen naturally in production: vendor API changes add new log levels, buggy agents emit out-of-range metrics, and upstream systems introduce nulls when fields aren't populated yet. Schema validation is your last line of defense before bad data enters your model.

In [ ]:
# Construct a DataFrame with intentional schema violations
bad_data = {
    "timestamp":        ["2024-01-15 08:00:00", "2024-01-15 08:00:30", "2024-01-15 08:01:00",
                          "2024-01-15 08:01:30", "2024-01-15 08:02:00"],
    "host":             ["ntnx-cvm-node-01", "ntnx-cvm-node-02", None,            # VIOLATION 3: null host
                          "ntnx-ahv-node-01", "ntnx-prism-central"],
    "log_level":        ["INFO", "CRITICAL", "INFO", "WARNING", "DEBUG"],           # VIOLATION 2: 'CRITICAL' not in schema
    "component":        ["AOS", "AHV", "Prism", "Stargate", "Cassandra"],
    "cpu_percent":      [45.2, 150.0, 62.1, 38.7, 71.4],                          # VIOLATION 1: 150.0 > 100
    "memory_mb":        [8192.0, 4096.5, 12288.0, 6144.0, 9216.0],
    "disk_io_mbps":     [25.3, 100.2, None, 55.8, 30.1],
    "response_time_ms": [120.5, 350.0, 90.2, None, 200.0],
}
bad_df = pd.DataFrame(bad_data)

print("Bad DataFrame (5 rows with 3 violations):")
bad_df

In [ ]:
# Attempt validation — this WILL fail, and that's the point
# Without lazy=True, Pandera raises on the FIRST error it encounters

print("Running schema.validate(bad_df) [eager mode — stops at first error]...\n")

try:
    ServerLogSchema.validate(bad_df)
except pa.errors.SchemaError as e:
    print(f"SchemaError caught!")
    print(f"\nError type: {type(e).__name__}")
    print(f"\nError message:\n{e}")

In [ ]:
# In production, you want to see ALL violations at once — not fix one at a time
# Use lazy=True to collect every failure before raising

print("Running schema.validate(bad_df, lazy=True) [collect ALL errors]...\n")

try:
    ServerLogSchema.validate(bad_df, lazy=True)
except pa.errors.SchemaErrors as e:
    print(f"SchemaErrors caught! Total violations: {len(e.failure_cases)}\n")
    print("Failure cases (each row = one violation):")
    print(e.failure_cases.to_string(index=False))
    print(f"\nFull error summary:")
    # Print the schema error detail table
    print(e.failure_cases[["schema_context", "column", "check", "check_number", "failure_case", "index"]].to_string(index=False))

### What Just Happened

Pandera identified all three violations simultaneously:
- **`cpu_percent` row 1**: value `150.0` failed the `less_than_or_equal_to(100.0)` check
- **`log_level` row 1**: value `"CRITICAL"` failed the `isin([...])` check
- **`host` row 2**: `None` failed the `not_nullable` constraint

In a real pipeline:

```python
# Production pattern: validate at ingestion boundary
try:
    validated_df = ServerLogSchema.validate(incoming_df, lazy=True)
    send_to_feature_store(validated_df)
except pa.errors.SchemaErrors as e:
    alert_on_call(f"Schema violation: {len(e.failure_cases)} errors")
    write_to_dead_letter_queue(incoming_df, e.failure_cases)
    raise  # Halt the pipeline — don't process corrupt data
```

This pattern is a **fail-fast** design. It is far better than letting bad data silently propagate to your model.

---

## Step 5: Writing a Data Contract

A **data contract** is a formal, machine-readable agreement between the producer of a dataset and its consumers. It goes beyond schema validation to include:

- **Ownership**: who is responsible for this data?
- **Business rules**: domain constraints that can't be expressed in types alone
- **SLAs**: freshness guarantees, update frequency
- **Versioning**: what changed between versions?

Think of it like a REST API contract — except for data. If the source system breaks the contract, it's an SLA violation, not just a bug.

> **Instructor Note:** This is a key concept for production ML. Discuss: at Nutanix, the team that owns AOS metrics data is different from the team building the anomaly detection model. Without a contract, silent schema changes from the infra team will silently break the ML pipeline. Contracts create accountability.

In [ ]:
# Define the data contract as a Python dict (will be serialized to JSON)
data_contract = {
    "contract_metadata": {
        "schema_version": "1.0.0",
        "contract_name": "nutanix_server_log_telemetry",
        "created_date": "2024-01-15",
        "last_updated": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "owner_team": "infrastructure-observability",
        "owner_email": "infra-observability@nutanix.com",
        "consumer_teams": ["ml-platform", "site-reliability", "capacity-planning"],
        "data_classification": "internal",
        "update_frequency": "every_30_seconds",
        "retention_days": 90
    },
    "schema": {
        "columns": [
            {"name": "timestamp",        "type": "string",  "nullable": False, "format": "YYYY-MM-DD HH:MM:SS", "description": "UTC timestamp"},
            {"name": "host",             "type": "string",  "nullable": False, "pattern": "^ntnx-[a-z]+-node-\\d+|ntnx-prism", "description": "Node hostname"},
            {"name": "log_level",        "type": "string",  "nullable": False, "allowed_values": ["ERROR", "WARNING", "INFO", "DEBUG"]},
            {"name": "component",        "type": "string",  "nullable": False, "description": "Nutanix software component"},
            {"name": "cpu_percent",      "type": "float",   "nullable": False, "min": 0.0, "max": 100.0},
            {"name": "memory_mb",        "type": "float",   "nullable": False, "min": 0.0},
            {"name": "disk_io_mbps",     "type": "float",   "nullable": True,  "min": 0.0},
            {"name": "response_time_ms", "type": "float",   "nullable": True,  "min": 0.0},
            {"name": "error_code",       "type": "string",  "nullable": True,  "description": "Optional error/warning code"}
        ]
    },
    "business_rules": [
        {
            "rule_id": "BR-001",
            "name": "cpu_physical_bounds",
            "description": "CPU utilization is a percentage and must be in [0, 100]",
            "column": "cpu_percent",
            "expression": "0.0 <= cpu_percent <= 100.0",
            "severity": "ERROR"
        },
        {
            "rule_id": "BR-002",
            "name": "no_future_timestamps",
            "description": "Log timestamps must not be in the future (data cannot arrive before it is generated)",
            "column": "timestamp",
            "expression": "timestamp <= now()",
            "severity": "ERROR"
        },
        {
            "rule_id": "BR-003",
            "name": "memory_reasonable_bound",
            "description": "Memory usage must not exceed 2TB (2,097,152 MB) — current max node spec",
            "column": "memory_mb",
            "expression": "memory_mb <= 2097152",
            "severity": "WARNING"
        },
        {
            "rule_id": "BR-004",
            "name": "dataset_freshness",
            "description": "Dataset must contain at least one record from the last 5 minutes",
            "column": "timestamp",
            "expression": "max(timestamp) >= now() - 5min  [only applicable in streaming context]",
            "severity": "WARNING"
        },
        {
            "rule_id": "BR-005",
            "name": "minimum_row_count",
            "description": "A batch must contain at least 10 rows to be considered valid (guards against empty/truncated feeds)",
            "column": "*",
            "expression": "len(df) >= 10",
            "severity": "ERROR"
        }
    ],
    "change_history": [
        {
            "version": "1.0.0",
            "date": "2024-01-15",
            "author": "infra-observability@nutanix.com",
            "description": "Initial contract for server log telemetry dataset",
            "breaking_change": False
        }
    ]
}

# Persist the contract as JSON
contract_path = os.path.join(LAB_DIR, "data_contract_server_logs.json")
with open(contract_path, "w") as f:
    json.dump(data_contract, f, indent=2)

print(f"Data contract saved to: {contract_path}")
print(f"\nContract summary:")
print(f"  Name:    {data_contract['contract_metadata']['contract_name']}")
print(f"  Version: {data_contract['contract_metadata']['schema_version']}")
print(f"  Owner:   {data_contract['contract_metadata']['owner_team']}")
print(f"  Columns: {len(data_contract['schema']['columns'])}")
print(f"  Rules:   {len(data_contract['business_rules'])}")

In [ ]:
def validate_against_contract(df: pd.DataFrame, contract: dict) -> dict:
    """
    Validate a DataFrame against a data contract specification.
    
    Checks:
      1. All contract-specified columns are present
      2. Nullable constraints are respected
      3. Allowed values for categorical columns
      4. Numeric min/max bounds
      5. Business rules (where programmatically checkable)
    
    Returns a validation report dict.
    """
    report = {
        "contract_name":    contract["contract_metadata"]["contract_name"],
        "schema_version":   contract["contract_metadata"]["schema_version"],
        "validated_at":     datetime.now(timezone.utc).isoformat(),
        "dataset_shape":    {"rows": len(df), "columns": len(df.columns)},
        "column_checks":    [],
        "business_rule_checks": [],
        "overall_status":   "PASS"
    }

    # ---- Column-level checks -----------------------------------------------
    for col_spec in contract["schema"]["columns"]:
        col_name  = col_spec["name"]
        col_result = {"column": col_name, "checks": [], "status": "PASS"}

        # Check presence
        if col_name not in df.columns:
            col_result["checks"].append({"check": "column_present", "passed": False, "detail": "Column missing from DataFrame"})
            col_result["status"] = "FAIL"
            report["column_checks"].append(col_result)
            report["overall_status"] = "FAIL"
            continue
        col_result["checks"].append({"check": "column_present", "passed": True, "detail": "OK"})

        # Check nullability
        if not col_spec.get("nullable", True):
            null_count = int(df[col_name].isnull().sum())
            passed = null_count == 0
            col_result["checks"].append({
                "check": "not_nullable", "passed": passed,
                "detail": "OK" if passed else f"{null_count} null values found"
            })
            if not passed:
                col_result["status"] = "FAIL"
                report["overall_status"] = "FAIL"

        # Check allowed_values (categorical columns)
        if "allowed_values" in col_spec:
            allowed = set(col_spec["allowed_values"])
            actual  = set(df[col_name].dropna().unique())
            unexpected = actual - allowed
            passed = len(unexpected) == 0
            col_result["checks"].append({
                "check": "allowed_values", "passed": passed,
                "detail": "OK" if passed else f"Unexpected values: {sorted(unexpected)}"
            })
            if not passed:
                col_result["status"] = "FAIL"
                report["overall_status"] = "FAIL"

        # Check numeric bounds
        if "min" in col_spec and pd.api.types.is_numeric_dtype(df[col_name]):
            below_min = int((df[col_name] < col_spec["min"]).sum())
            passed = below_min == 0
            col_result["checks"].append({
                "check": f"min_value_{col_spec['min']}", "passed": passed,
                "detail": "OK" if passed else f"{below_min} rows below minimum {col_spec['min']}"
            })
            if not passed:
                col_result["status"] = "FAIL"
                report["overall_status"] = "FAIL"

        if "max" in col_spec and pd.api.types.is_numeric_dtype(df[col_name]):
            above_max = int((df[col_name] > col_spec["max"]).sum())
            passed = above_max == 0
            col_result["checks"].append({
                "check": f"max_value_{col_spec['max']}", "passed": passed,
                "detail": "OK" if passed else f"{above_max} rows above maximum {col_spec['max']}"
            })
            if not passed:
                col_result["status"] = "FAIL"
                report["overall_status"] = "FAIL"

        report["column_checks"].append(col_result)

    # ---- Business rule checks (programmatically evaluable ones) ------------
    for rule in contract["business_rules"]:
        rule_result = {"rule_id": rule["rule_id"], "name": rule["name"], "severity": rule["severity"]}

        if rule["rule_id"] == "BR-001" and "cpu_percent" in df.columns:
            invalid = int(((df["cpu_percent"] < 0) | (df["cpu_percent"] > 100)).sum())
            rule_result["passed"] = invalid == 0
            rule_result["detail"] = "OK" if invalid == 0 else f"{invalid} cpu_percent values out of [0, 100]"

        elif rule["rule_id"] == "BR-003" and "memory_mb" in df.columns:
            over = int((df["memory_mb"] > 2_097_152).sum())
            rule_result["passed"] = over == 0
            rule_result["detail"] = "OK" if over == 0 else f"{over} rows exceed 2TB memory bound"

        elif rule["rule_id"] == "BR-005":
            passed = len(df) >= 10
            rule_result["passed"] = passed
            rule_result["detail"] = f"Row count {len(df)} >= 10" if passed else f"Only {len(df)} rows — below minimum"

        else:
            # Rules that require runtime context (streaming freshness, future-timestamp check)
            rule_result["passed"] = None
            rule_result["detail"] = "Skipped — requires runtime context (streaming/real-time check)"

        if rule_result.get("passed") is False and rule["severity"] == "ERROR":
            report["overall_status"] = "FAIL"

        report["business_rule_checks"].append(rule_result)

    return report


# Run validation
contract_report = validate_against_contract(df, data_contract)

# Display the report
print("=" * 70)
print(f" CONTRACT VALIDATION REPORT")
print(f" Contract: {contract_report['contract_name']}  v{contract_report['schema_version']}")
print(f" Validated: {contract_report['validated_at']}")
print(f" Dataset: {contract_report['dataset_shape']['rows']} rows x {contract_report['dataset_shape']['columns']} cols")
print("=" * 70)

print("\n--- Column Checks ---")
for col_check in contract_report["column_checks"]:
    status_icon = "✓" if col_check["status"] == "PASS" else "✗"
    print(f"  [{status_icon}] {col_check['column']:20s}  [{col_check['status']}]")
    for c in col_check["checks"]:
        ok = "✓" if c["passed"] else "✗"
        print(f"       {ok}  {c['check']:35s}  {c['detail']}")

print("\n--- Business Rule Checks ---")
for rule_check in contract_report["business_rule_checks"]:
    passed = rule_check.get("passed")
    if passed is True:
        icon = "✓"
    elif passed is False:
        icon = "✗"
    else:
        icon = "~"
    print(f"  [{icon}] {rule_check['rule_id']}  {rule_check['name']:30s}  {rule_check['detail']}")

print("=" * 70)
overall = contract_report["overall_status"]
print(f"  OVERALL STATUS: {overall}")
print("=" * 70)

---

## Step 6: Introduction to DVC — Data Version Control

### What is DVC?

**DVC (Data Version Control)** is an open-source tool that adds Git-like versioning for large data files, datasets, and ML models. It stores data externally (S3, GCS, Azure Blob, local NAS) and tracks *references* to that data in Git.

### The Core Problem DVC Solves

Git is not designed for large binary files. A 10GB training dataset cannot be committed to a Git repo. But you still need to know:
- Which version of the data was used to train model v2.3?
- Who modified the dataset last week, and what changed?
- How do I roll back to the dataset from last month?

DVC solves this by:
1. Storing the data in remote storage (S3/GCS/etc.)
2. Committing a small `.dvc` pointer file to Git (contains hash, size, path)
3. Providing `dvc pull`/`dvc push` to sync data to/from remote storage

> **Instructor Note:** Draw the architecture on the whiteboard: Git repo (code + .dvc files) ↔ remote storage (actual data). Emphasize: the `.dvc` file IS the version — it's a tiny JSON that Git tracks. When you `git checkout` an old commit, `dvc pull` fetches the corresponding data version.

### DVC Workflow — Command Reference

The following commands represent a complete DVC workflow. In a demo environment with DVC installed (`pip install dvc dvc-s3`), you would run these in your terminal:

```bash
# ---- ONE-TIME SETUP ----

# Initialize DVC in your Git repository
dvc init
# Creates: .dvc/ directory, .dvc/.gitignore, .dvcignore

# Configure remote storage (S3 bucket at Nutanix)
dvc remote add -d nutanix-ml-data s3://nutanix-ml-platform/datasets/
dvc remote modify nutanix-ml-data region us-west-2

# ---- VERSIONING A DATASET ----

# Track the cleaned data file — DVC computes its hash and creates a .dvc pointer
dvc add data/cleaned_server_logs.csv
# Creates: data/cleaned_server_logs.csv.dvc  (commit this to Git)
# Adds:    data/cleaned_server_logs.csv to .gitignore  (don't commit the actual data)

# Commit the pointer file to Git
git add data/cleaned_server_logs.csv.dvc data/.gitignore
git commit -m "feat: track server log dataset v1.0 with DVC"

# Upload the actual data to S3
dvc push

# ---- COLLABORATING ----

# A teammate clones the repo and fetches data
git clone https://github.com/nutanix/ml-platform.git
dvc pull   # downloads data matching the .dvc pointer in the current commit

# ---- ROLLING BACK TO A PREVIOUS DATA VERSION ----

# Check out a previous Git commit (the .dvc file changes)
git checkout abc1234  # the commit where data v1.0 was tracked
dvc checkout          # DVC restores the data file matching that .dvc pointer

# ---- VIEWING PIPELINE STATE ----

# Show data file status (modified? cached? remote?)
dvc status

# Show the DAG of a DVC pipeline (if using dvc.yaml)
dvc dag

# Show metrics differences between Git commits
dvc metrics diff HEAD~1
```

### What a `.dvc` Pointer File Looks Like

```yaml
# data/cleaned_server_logs.csv.dvc
outs:
- md5: d41f8cd98f00b204e9800998ecf8427e
  size: 48291
  path: cleaned_server_logs.csv
```

The `md5` hash is the version identifier. Git diffs on this file tell you exactly what data changed.

### DVC vs. Git LFS vs. MLflow Artifacts

| Feature | DVC | Git LFS | MLflow Artifacts |
|---|---|---|---|
| Large files | Yes | Yes | Yes |
| Multiple backends | Yes (S3/GCS/Azure/local) | Limited | Yes |
| Pipeline orchestration | Yes (dvc.yaml) | No | No |
| Experiment tracking | Yes (dvc metrics) | No | Yes |
| Free for private repos | Yes | No (GitHub) | Yes |
| ML-native | Yes | No | Yes |

---

## Step 7: Lightweight Data Versioning Without DVC

When DVC isn't available (e.g., a quick prototyping environment, a local laptop, or a team that hasn't adopted DVC yet), you can implement a simple versioning system in pure Python.

The key insight: **a version is just a hash + metadata**. We compute the MD5 of the file content, save it with a timestamped name, and maintain a registry JSON.

This is a stepping stone, not a replacement for DVC — but it demonstrates the *concepts* of content-addressed storage and version registries.

> **Instructor Note:** Emphasize the engineering principle: content-addressed storage (the same idea behind Git objects, Docker layers, and DVC). If the hash is the same, the content is the same. If the hash changes, something changed.

In [ ]:
# Registry file path — tracks all dataset versions produced in this lab
REGISTRY_PATH = os.path.join(LAB_DIR, "dataset_registry.json")


def compute_dataframe_hash(df: pd.DataFrame) -> str:
    """Compute an MD5 hash of a DataFrame's CSV representation."""
    csv_bytes = df.to_csv(index=False).encode("utf-8")
    return hashlib.md5(csv_bytes).hexdigest()


def load_registry(registry_path: str) -> dict:
    """Load the version registry from disk, or return an empty one."""
    if os.path.exists(registry_path):
        with open(registry_path, "r") as f:
            return json.load(f)
    return {"dataset_versions": []}


def version_dataset(df: pd.DataFrame, dataset_name: str, description: str = "") -> dict:
    """
    Version a DataFrame:
      1. Compute content hash
      2. Assign incremental version number
      3. Save CSV with versioned filename: <name>_v<N>_<YYYYMMDD>_<hash8>.csv
      4. Append version entry to the registry JSON
    
    Returns the version metadata dict.
    """
    registry = load_registry(REGISTRY_PATH)

    # Filter versions for this dataset name
    existing = [v for v in registry["dataset_versions"] if v["dataset_name"] == dataset_name]
    version_num = len(existing) + 1

    content_hash = compute_dataframe_hash(df)
    short_hash   = content_hash[:8]
    timestamp    = datetime.now(timezone.utc)
    date_str     = timestamp.strftime("%Y%m%d_%H%M%S")

    # Check if identical content already exists
    for v in existing:
        if v["content_hash"] == content_hash:
            print(f"[INFO] Identical content already registered as version {v['version']}. Skipping save.")
            return v

    filename     = f"{dataset_name}_v{version_num}_{date_str}_{short_hash}.csv"
    save_path    = os.path.join(LAB_DIR, filename)

    df.to_csv(save_path, index=False)

    version_entry = {
        "dataset_name":     dataset_name,
        "version":          version_num,
        "version_tag":      f"v{version_num}",
        "filename":         filename,
        "path":             save_path,
        "content_hash":     content_hash,
        "short_hash":       short_hash,
        "created_at":       timestamp.isoformat(),
        "shape":            {"rows": len(df), "columns": len(df.columns)},
        "columns":          list(df.columns),
        "description":      description,
        "size_bytes":       os.path.getsize(save_path)
    }

    registry["dataset_versions"].append(version_entry)

    with open(REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)

    print(f"[VERSIONED] {dataset_name} {version_entry['version_tag']}")
    print(f"  File:    {filename}")
    print(f"  Hash:    {content_hash}")
    print(f"  Shape:   {version_entry['shape']}")
    print(f"  Size:    {version_entry['size_bytes']:,} bytes")
    return version_entry


# Version the current clean dataset
print("=== Versioning Initial Clean Dataset ===")
v1_entry = version_dataset(
    df,
    dataset_name="server_logs",
    description="Initial cleaned Nutanix server log telemetry — 500 rows, output of Lab 1.2"
)

In [ ]:
# Simulate a data change: filter to ERROR and WARNING only
# This represents a common pipeline step: creating a "high-severity events" subset

df_high_severity = df[df["log_level"].isin(["ERROR", "WARNING"])].reset_index(drop=True)
df_high_severity["severity_tier"] = "HIGH"  # Add a derived column

print(f"High-severity subset: {len(df_high_severity)} rows (from {len(df)} original rows)")
print(f"log_level distribution:\n{df_high_severity['log_level'].value_counts().to_string()}")

print("\n=== Versioning Modified Dataset ===")
v2_entry = version_dataset(
    df_high_severity,
    dataset_name="server_logs",
    description="High-severity events only (ERROR + WARNING), added severity_tier column — for anomaly detection training"
)

In [ ]:
# Show the complete version registry
registry = load_registry(REGISTRY_PATH)

print("=" * 70)
print(" DATASET VERSION REGISTRY")
print("=" * 70)

for v in registry["dataset_versions"]:
    print(f"\n  {v['dataset_name']}  {v['version_tag']}")
    print(f"    Hash:        {v['content_hash']}")
    print(f"    Created:     {v['created_at']}")
    print(f"    Shape:       {v['shape']['rows']} rows x {v['shape']['columns']} cols")
    print(f"    File:        {v['filename']}")
    print(f"    Description: {v['description']}")

print("\n" + "=" * 70)
print(f"  Total versions tracked: {len(registry['dataset_versions'])}")
print("=" * 70)

---

## Step 8: Data Lineage Tracking

**Data lineage** is the audit trail of how data moved and transformed from its source to its final form. In production ML systems, lineage answers questions like:

- "The model's precision dropped — which data transformation introduced this?"
- "A compliance audit requires us to show every transformation applied to PII data"
- "We need to re-run the pipeline from Step 3 only — what does Step 3 depend on?"

We'll build a simple `DataLineage` class that tracks each transformation step: what went in, what came out, and what was done.

> **Instructor Note:** Compare to a transaction log in a database — every operation is recorded. Modern data stacks (dbt, Apache Atlas, OpenLineage) formalize this, but the concept is the same. Engineers building pipelines should record lineage as a first-class concern, not an afterthought.

In [ ]:
class DataLineage:
    """
    Lightweight data lineage tracker for ML pipeline steps.
    
    Records the chain of transformations from raw source data
    to the final processed dataset used for model training.
    
    Each step records:
      - Step name and description
      - Row/column counts before and after
      - Transformation type (filter, enrich, aggregate, validate, ...)
      - Timestamp
      - Optional metadata (e.g., parameters used)
    """

    def __init__(self, pipeline_name: str, source_description: str):
        self.pipeline_name = pipeline_name
        self.created_at    = datetime.now(timezone.utc).isoformat()
        self.steps         = []
        self._add_source(source_description)

    def _add_source(self, description: str):
        self.steps.append({
            "step_number":     0,
            "step_name":       "SOURCE",
            "step_type":       "source",
            "description":     description,
            "rows_in":         None,
            "rows_out":        None,
            "cols_in":         None,
            "cols_out":        None,
            "rows_delta":      None,
            "timestamp":       self.created_at,
            "metadata":        {}
        })

    def add_step(
        self,
        name: str,
        description: str,
        rows_before: int,
        rows_after: int,
        cols_before: int = None,
        cols_after: int = None,
        step_type: str = "transform",
        metadata: dict = None
    ):
        """Record a single transformation step."""
        step_number = len([s for s in self.steps if s["step_type"] != "source"]) + 1
        self.steps.append({
            "step_number":  step_number,
            "step_name":    name,
            "step_type":    step_type,
            "description":  description,
            "rows_in":      rows_before,
            "rows_out":     rows_after,
            "cols_in":      cols_before,
            "cols_out":     cols_after,
            "rows_delta":   rows_after - rows_before,
            "timestamp":    datetime.now(timezone.utc).isoformat(),
            "metadata":     metadata or {}
        })
        return self

    def save_lineage(self, path: str):
        """Persist lineage report to JSON."""
        lineage_doc = {
            "pipeline_name":  self.pipeline_name,
            "created_at":     self.created_at,
            "total_steps":    len([s for s in self.steps if s["step_type"] != "source"]),
            "steps":          self.steps
        }
        with open(path, "w") as f:
            json.dump(lineage_doc, f, indent=2)
        print(f"Lineage report saved to: {path}")
        return lineage_doc

    def print_summary(self):
        """Print a human-readable lineage summary."""
        print("=" * 70)
        print(f" DATA LINEAGE — {self.pipeline_name}")
        print(f" Created: {self.created_at}")
        print("=" * 70)

        for step in self.steps:
            if step["step_type"] == "source":
                print(f"  [SOURCE] {step['description']}")
                print(f"           {'─' * 50}")
                continue

            delta_str = ""
            if step["rows_delta"] is not None:
                delta_str = f" ({step['rows_delta']:+d} rows)"

            col_str = ""
            if step["cols_in"] is not None and step["cols_out"] is not None:
                col_str = f"  {step['cols_in']}→{step['cols_out']} cols"

            print(f"  Step {step['step_number']:2d} [{step['step_type'].upper():10s}]  {step['step_name']}")
            print(f"         {step['description']}")
            print(f"         Rows: {step['rows_in']} → {step['rows_out']}{delta_str}{col_str}")
            if step["metadata"]:
                for k, v in step["metadata"].items():
                    print(f"         {k}: {v}")
            print(f"         {'─' * 50}")

        transform_steps = [s for s in self.steps if s["step_type"] != "source"]
        if transform_steps:
            first = transform_steps[0]
            last  = transform_steps[-1]
            total_delta = (last["rows_out"] or 0) - (first["rows_in"] or 0)
            print(f"\n  Pipeline summary: {first['rows_in']} rows in → {last['rows_out']} rows out ({total_delta:+d} rows)")
        print("=" * 70)


print("DataLineage class defined.")

In [ ]:
# Walk through the FULL Module 1 pipeline lineage
# This traces the entire journey from raw logs to ML-ready features

lineage = DataLineage(
    pipeline_name="nutanix_server_logs_module1_pipeline",
    source_description="Raw Nutanix cluster node event logs — NTNX-AOS syslog export (CSV)"
)

# Step 1: Initial ingestion (Lab 1.1 work)
lineage.add_step(
    name="raw_ingestion",
    description="Load raw syslog CSV; cast timestamp column to datetime; drop malformed lines",
    rows_before=612,
    rows_after=608,
    cols_before=9,
    cols_after=9,
    step_type="ingest",
    metadata={"source_file": "raw_server_logs.csv", "malformed_rows_dropped": 4}
)

# Step 2: Deduplication (Lab 1.2)
lineage.add_step(
    name="deduplication",
    description="Remove duplicate rows based on (timestamp, host, component) composite key",
    rows_before=608,
    rows_after=598,
    cols_before=9,
    cols_after=9,
    step_type="clean",
    metadata={"duplicates_removed": 10, "dedup_key": "timestamp + host + component"}
)

# Step 3: Missing value imputation (Lab 1.2)
lineage.add_step(
    name="missing_value_imputation",
    description="Impute disk_io_mbps with median per host; drop rows where cpu_percent is null",
    rows_before=598,
    rows_after=500,
    cols_before=9,
    cols_after=9,
    step_type="clean",
    metadata={
        "disk_io_imputed_with": "median_per_host",
        "rows_dropped_null_cpu": 98
    }
)

# Step 4: Feature engineering (Lab 1.3)
lineage.add_step(
    name="feature_engineering",
    description="Add: hour_of_day, is_weekend, cpu_memory_ratio, rolling_avg_cpu_5min, error_flag, log_level_encoded",
    rows_before=500,
    rows_after=500,
    cols_before=9,
    cols_after=15,
    step_type="enrich",
    metadata={
        "new_features": ["hour_of_day", "is_weekend", "cpu_memory_ratio",
                          "rolling_avg_cpu_5min", "error_flag", "log_level_encoded"],
        "features_removed": []
    }
)

# Step 5: Schema validation (this lab)
lineage.add_step(
    name="schema_validation",
    description="Pandera schema validation against ServerLogSchema v1.0.0; data contract check",
    rows_before=500,
    rows_after=500,
    cols_before=15,
    cols_after=15,
    step_type="validate",
    metadata={
        "schema": "ServerLogTelemetrySchema",
        "contract_version": "1.0.0",
        "validation_result": "PASS",
        "rows_rejected": 0
    }
)

# Step 6: Output to feature store (final step)
lineage.add_step(
    name="feature_store_write",
    description="Write ML-ready feature DataFrame to versioned feature store partition",
    rows_before=500,
    rows_after=500,
    cols_before=15,
    cols_after=15,
    step_type="output",
    metadata={
        "output_path": "s3://nutanix-ml-platform/features/server_logs/20240115/",
        "format": "parquet",
        "partition_key": "date"
    }
)

# Print the lineage
lineage.print_summary()

# Save to JSON
lineage_path = os.path.join(LAB_DIR, "lineage_report.json")
lineage_doc  = lineage.save_lineage(lineage_path)

---

## Step 9: Generating a Data Quality Report

A data quality report packages all the checks we've done into a single sharable artifact. In production, this report would be generated on every pipeline run and stored alongside the dataset version.

We'll generate:
1. A structured JSON summary
2. A missing-values visualization
3. Distribution plots for key numeric columns

> **Instructor Note:** In production systems, this report would be attached to the training run metadata (e.g., in MLflow Experiments) so anyone can trace which data quality state was present when a particular model version was trained.

In [ ]:
def generate_quality_report(df: pd.DataFrame, schema=None, save_dir: str = None) -> dict:
    """
    Generate a comprehensive data quality report for a DataFrame.
    
    Produces:
      - Structural summary (rows, cols, dtypes)
      - Missing value analysis (per column)
      - Numeric column statistics
      - Categorical column distributions
      - Schema validation result (if schema provided)
      - Visualizations (missing %, distributions)
    """
    report = {
        "generated_at":    datetime.now(timezone.utc).isoformat(),
        "structure": {
            "total_rows":    len(df),
            "total_columns": len(df.columns),
            "dtypes":        df.dtypes.astype(str).to_dict()
        },
        "missing_values": {},
        "numeric_stats":  {},
        "categorical_distributions": {},
        "schema_validation": None
    }

    # Missing values analysis
    for col in df.columns:
        null_count  = int(df[col].isnull().sum())
        null_pct    = round(null_count / len(df) * 100, 2)
        report["missing_values"][col] = {
            "null_count":   null_count,
            "null_percent": null_pct,
            "complete":     null_count == 0
        }

    # Numeric stats
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        s = df[col].dropna()
        report["numeric_stats"][col] = {
            "count":  int(s.count()),
            "mean":   round(float(s.mean()), 4),
            "std":    round(float(s.std()), 4),
            "min":    round(float(s.min()), 4),
            "p25":    round(float(s.quantile(0.25)), 4),
            "median": round(float(s.median()), 4),
            "p75":    round(float(s.quantile(0.75)), 4),
            "max":    round(float(s.max()), 4),
            "zeros":  int((s == 0).sum())
        }

    # Categorical distributions
    cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
    for col in cat_cols:
        vc = df[col].value_counts(dropna=False).head(20)
        report["categorical_distributions"][col] = vc.to_dict()

    # Schema validation
    if schema is not None:
        try:
            schema.validate(df, lazy=True)
            report["schema_validation"] = {"status": "PASS", "errors": []}
        except pa.errors.SchemaErrors as e:
            errors = e.failure_cases[["column", "check", "failure_case", "index"]].to_dict(orient="records")
            report["schema_validation"] = {"status": "FAIL", "errors": errors[:50]}  # Cap at 50 errors
        except pa.errors.SchemaError as e:
            report["schema_validation"] = {"status": "FAIL", "errors": [str(e)]}

    # ---- Visualizations ----
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Data Quality Report — Nutanix Server Log Telemetry", fontsize=14, fontweight="bold")

    # Plot 1: Missing values per column
    ax = axes[0, 0]
    miss_df = pd.DataFrame([
        {"column": col, "null_percent": report["missing_values"][col]["null_percent"]}
        for col in df.columns
    ]).sort_values("null_percent", ascending=True)
    colors = ["#d62728" if p > 5 else "#2ca02c" for p in miss_df["null_percent"]]
    bars = ax.barh(miss_df["column"], miss_df["null_percent"], color=colors)
    ax.set_xlabel("Missing %")
    ax.set_title("Missing Values by Column")
    ax.axvline(x=5, color="orange", linestyle="--", alpha=0.7, label="5% threshold")
    ax.legend(fontsize=8)
    for bar, val in zip(bars, miss_df["null_percent"]):
        if val > 0:
            ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
                    f"{val:.1f}%", va='center', fontsize=8)

    # Plot 2: CPU utilization distribution
    ax = axes[0, 1]
    cpu_data = df["cpu_percent"].dropna()
    ax.hist(cpu_data, bins=30, color="#1f77b4", edgecolor="white", alpha=0.8)
    ax.axvline(cpu_data.mean(), color="red", linestyle="--", label=f"Mean: {cpu_data.mean():.1f}%")
    ax.axvline(cpu_data.median(), color="orange", linestyle="--", label=f"Median: {cpu_data.median():.1f}%")
    ax.set_xlabel("CPU %")
    ax.set_ylabel("Frequency")
    ax.set_title("CPU Utilization Distribution")
    ax.legend(fontsize=9)

    # Plot 3: Memory usage distribution
    ax = axes[0, 2]
    mem_data = df["memory_mb"].dropna()
    ax.hist(mem_data, bins=30, color="#ff7f0e", edgecolor="white", alpha=0.8)
    ax.axvline(mem_data.mean(), color="red", linestyle="--", label=f"Mean: {mem_data.mean():.0f} MB")
    ax.set_xlabel("Memory (MB)")
    ax.set_ylabel("Frequency")
    ax.set_title("Memory Usage Distribution")
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1024:.0f}GB"))

    # Plot 4: Log level distribution (categorical)
    ax = axes[1, 0]
    level_counts = df["log_level"].value_counts()
    level_colors = {"ERROR": "#d62728", "WARNING": "#ff7f0e", "INFO": "#2ca02c", "DEBUG": "#9467bd"}
    bar_colors   = [level_colors.get(l, "#7f7f7f") for l in level_counts.index]
    ax.bar(level_counts.index, level_counts.values, color=bar_colors, edgecolor="white")
    ax.set_xlabel("Log Level")
    ax.set_ylabel("Count")
    ax.set_title("Log Level Distribution")
    for i, (idx, val) in enumerate(level_counts.items()):
        ax.text(i, val + 2, str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Plot 5: Response time distribution (log scale)
    ax = axes[1, 1]
    rt_data = df["response_time_ms"].dropna()
    ax.hist(rt_data, bins=40, color="#17becf", edgecolor="white", alpha=0.8)
    ax.set_xlabel("Response Time (ms)")
    ax.set_ylabel("Frequency")
    ax.set_title("Response Time Distribution")
    ax.axvline(rt_data.quantile(0.95), color="red", linestyle="--",
               label=f"P95: {rt_data.quantile(0.95):.0f}ms")
    ax.legend(fontsize=9)

    # Plot 6: Top 6 components by event count
    ax = axes[1, 2]
    comp_counts = df["component"].value_counts().head(6)
    ax.barh(comp_counts.index[::-1], comp_counts.values[::-1],
            color="#8c564b", edgecolor="white", alpha=0.85)
    ax.set_xlabel("Event Count")
    ax.set_title("Top 6 Components by Event Count")
    for bar, val in zip(ax.patches, comp_counts.values[::-1]):
        ax.text(val + 1, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=9)

    plt.tight_layout()

    if save_dir:
        plot_path = os.path.join(save_dir, "data_quality_plots.png")
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        print(f"Quality plots saved to: {plot_path}")
        report["plot_path"] = plot_path

    plt.show()
    return report


# Generate the report
quality_report = generate_quality_report(df, schema=ServerLogSchema, save_dir=LAB_DIR)

In [ ]:
# Save the quality report JSON
report_path = os.path.join(LAB_DIR, "data_quality_report.json")

# Convert any non-serializable keys (None keys from value_counts) before saving
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {(str(k) if k is None else k): make_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_safe(i) for i in obj]
    elif isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    else:
        return obj

safe_report = make_json_safe(quality_report)

with open(report_path, "w") as f:
    json.dump(safe_report, f, indent=2)

print(f"Quality report saved to: {report_path}")

# Print the text summary
print("\n" + "=" * 70)
print(" DATA QUALITY REPORT SUMMARY")
print("=" * 70)
print(f"  Generated:     {quality_report['generated_at']}")
print(f"  Rows:          {quality_report['structure']['total_rows']:,}")
print(f"  Columns:       {quality_report['structure']['total_columns']}")

# Missing values summary
mv = quality_report["missing_values"]
cols_with_nulls = [(c, v) for c, v in mv.items() if v["null_count"] > 0]
print(f"\n  Missing Values:")
if cols_with_nulls:
    for col, v in sorted(cols_with_nulls, key=lambda x: -x[1]["null_percent"]):
        print(f"    {col:25s}  {v['null_count']:4d} nulls  ({v['null_percent']:.1f}%)")
else:
    print("    All required columns: 100% complete")

# Numeric summary
print(f"\n  Numeric Column Quick Stats:")
for col, stats in quality_report["numeric_stats"].items():
    print(f"    {col:25s}  mean={stats['mean']:>10.2f}  std={stats['std']:>9.2f}  min={stats['min']:>8.2f}  max={stats['max']:>10.2f}")

# Schema validation
sv = quality_report["schema_validation"]
if sv:
    status_icon = "✓" if sv["status"] == "PASS" else "✗"
    print(f"\n  Schema Validation: [{status_icon}] {sv['status']}")
    if sv["errors"]:
        print(f"    {len(sv['errors'])} errors found")

print("=" * 70)

---

## Lab Checkpoint — Discussion Questions

Take 5–7 minutes. Discuss with your pair or table:

**1. Schema Validation Placement**
> Where in your current team's pipeline would schema validation prevent the most bugs? Is it at ingestion, after cleaning, before model training, or at inference time? Can it be more than one place?

**2. The Data Contract as an Organizational Tool**
> A data contract only works if both the producer team and the consumer team agree to it. In your organization, which teams would own the contract for infrastructure telemetry data? What happens when the contract is violated — is it a bug, an incident, or a feature request?

**3. Versioning Trade-offs**
> The lightweight Python versioning we built saves full CSV copies per version. At Nutanix scale (multi-TB datasets), this is impractical. What are the design trade-offs between: storing full copies vs. storing diffs vs. using content-addressed storage (DVC/Git LFS)?

**4. Lineage and Debugging**
> Imagine your anomaly detection model's precision drops by 12% after a pipeline run. You have lineage tracking. Walk through how you'd use the lineage report to isolate the step where data quality degraded. What information would you add to the lineage tracker to make this debugging faster?

---

## Lab Artifact Summary

The following files were produced in this lab:

In [ ]:
# Show all artifacts produced in this lab
artifacts = [
    ("cleaned_server_logs.csv",           "Base dataset (from Lab 1.2 or generated)"),
    ("data_contract_server_logs.json",     "Data contract v1.0.0 for server log telemetry"),
    ("dataset_registry.json",             "Dataset version registry (2 versions tracked)"),
    ("lineage_report.json",               "Full pipeline lineage for Module 1"),
    ("data_quality_report.json",          "Structured data quality report"),
    ("data_quality_plots.png",            "Quality report visualizations"),
]

print("=" * 70)
print(" LAB 1.4 ARTIFACTS")
print("=" * 70)
for filename, description in artifacts:
    path = os.path.join(LAB_DIR, filename)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path):>8,} bytes" if exists else "       not found"
    icon   = "✓" if exists else "○"
    print(f"  [{icon}] {filename:45s} {size}")
    print(f"       {description}")
print("=" * 70)

---

## Key Takeaways

### What You Built in This Lab

| Capability | Tool / Technique | Production Equivalent |
|---|---|---|
| Manual data checks | Custom Python assertions | Unit tests for data |
| Declarative schema validation | Pandera `DataFrameSchema` | Great Expectations, dbt tests |
| Data contract | JSON schema + business rules | Data mesh contracts, OpenAPI for data |
| Data versioning (lightweight) | Hash + registry JSON | DVC, Delta Lake, Iceberg |
| Data versioning (production) | DVC + remote storage | MLflow Data, Feast |
| Data lineage | `DataLineage` class | Apache Atlas, OpenLineage, dbt lineage |
| Quality report | `generate_quality_report()` | Monte Carlo, Datafold, dbt docs |

### The Core Engineering Principle

**Treat data as a first-class engineering artifact**, not an afterthought.

Just as you would not deploy code without tests, you should not train a model without:
- A validated schema
- A versioned dataset
- A recorded lineage trail
- A quality report

These aren't bureaucratic overhead — they are the **debugging surface** you'll be grateful for at 2am when a production model degrades.

---

### Module 1 Complete

Congratulations — you've completed all four labs of Module 1:

| Lab | Topic | Status |
|---|---|---|
| Lab 1.1 | Data Exploration & EDA | Complete |
| Lab 1.2 | Data Cleaning & Preprocessing | Complete |
| Lab 1.3 | Feature Engineering | Complete |
| **Lab 1.4** | **Data Validation, Contracts & Versioning** | **Complete** |

---

### Preview: Module 2

In **Module 2: Model Development & Experimentation**, you will:

- Build baseline ML models on the validated, versioned data from Module 1
- Use **MLflow** for experiment tracking — logging parameters, metrics, and artifacts
- Implement cross-validation and hyperparameter search
- Compare model versions systematically
- Introduction to **AutoML** and when to use it

The data contract and versioning work you did today will directly connect to model experiment tracking — your training runs will reference specific dataset versions, creating end-to-end reproducibility.

> **Instructor Note:** End with a quick recap question: "If I asked you to reproduce your best model from 3 months ago — what would you need to be able to do that?" The answer encompasses everything in Module 1: the exact code (Git), the exact data (DVC/versioning), the exact schema (contract), and the exact environment (Docker/conda). Module 2 adds: the exact hyperparameters and metrics (MLflow).

---
## 🎯 Your Turn — Challenges

These challenges extend the validation and versioning work from this lab.  
Use the `df`, `schema`, and `contract` objects already defined above.

### Challenge 1 — Extend the Data Contract

The current contract has 5 business rules (BR-001 to BR-005).

**Task:**  
Add 2 new business rules to the contract dict:
- `BR-006`: `response_time_ms` must be less than **5000ms** (5-second SLA breach threshold)
- `BR-007`: No single `host` should appear in more than **30%** of all rows (detects data skew / collection bias)

Then update `validate_against_contract()` to check these new rules and re-run the validation report.

*Hint: For BR-007 use `df['host'].value_counts(normalize=True).max()`*

In [ ]:
# Challenge 1 — Your solution here




### Challenge 2 — Dataset Diff Report

When you version a dataset, you want to know **what changed** between versions.

**Task:**  
Write a function `diff_datasets(df_v1, df_v2, name="dataset")` that returns a dict with:
- `row_count_change`: int (v2 rows - v1 rows)
- `new_columns`: list of columns in v2 but not v1
- `dropped_columns`: list of columns in v1 but not v2
- `null_change`: dict of `{column: (v1_null_pct, v2_null_pct)}` for columns where null % changed by more than 2%
- `dtype_changes`: dict of `{column: (v1_dtype, v2_dtype)}` for any type changes

Test it by:
1. Creating `df_v2` from `df` with: one column dropped, one new column added, and 10 extra null values introduced in `cpu_percent`
2. Running `diff_datasets(df, df_v2)`
3. Printing the report in a readable format

In [ ]:
# Challenge 2 — Your solution here




### Challenge 3 — Pandera Custom Check

Pandera supports custom business-logic checks beyond simple range checks.

**Task:**  
Add a **custom Pandera check** to the schema that validates:  
*"For any row where `log_level` is `ERROR`, the `cpu_percent` must be greater than 50 OR `response_time_ms` must be greater than 200."*

This encodes domain knowledge: an ERROR log with normal CPU and fast response is suspicious and likely a data quality issue.

Steps:
1. Define the check using `pa.Check` with a lambda or custom function
2. Apply it as a DataFrame-level check (not column-level) using `checks=` in `DataFrameSchema`
3. Create a test DataFrame that violates this rule and verify the schema catches it

*Hint: `pa.DataFrameSchema(checks=[pa.Check(lambda df: ...)])`*

In [ ]:
# Challenge 3 — Your solution here


